In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [12]:
import os
print(os.getcwd())


/content


In [13]:
cd /content/drive/MyDrive/deepfake_voice_iot


/content/drive/MyDrive/deepfake_voice_iot


In [14]:
import numpy as np

X_train = np.load("data/X_train.npy")
y_train = np.load("data/y_train.npy")

X_dev = np.load("data/X_dev.npy")
y_dev = np.load("data/y_dev.npy")

print(X_train.shape, y_train.shape)
print(X_dev.shape, y_dev.shape)


(25380, 154, 126) (25380,)
(24844, 154, 126) (24844,)


In [15]:
X_train = X_train[..., np.newaxis]
X_dev = X_dev[..., np.newaxis]

print(X_train.shape)
print(X_dev.shape)


(25380, 154, 126, 1)
(24844, 154, 126, 1)


In [16]:
import tensorflow as tf
print(tf.__version__)

print("GPU available:", tf.config.list_physical_devices('GPU'))


2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [17]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(154,126,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 152, 124, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 152, 124, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 76, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 74, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 74, 60, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 35, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 35, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 30464)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,899,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,993,217 (15.23 MB)

 Trainable params: 3,992,769 (15.23 MB)

 Non-trainable params: 448 (1.75 KB)

In [18]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_dev, y_dev)
)


Epoch 1/10
794/794 ━━━━━━━━━━━━━━━━━━━━ 48s 48ms/step - accuracy: 0.8891 - loss: 0.5109 - val_accuracy: 0.9323 - val_loss: 0.1591
Epoch 2/10
794/794 ━━━━━━━━━━━━━━━━━━━━ 22s 28ms/step - accuracy: 0.9529 - loss: 0.1267 - val_accuracy: 0.9571 - val_loss: 0.1289
Epoch 3/10
794/794 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.9798 - loss: 0.0766 - val_accuracy: 0.9564 - val_loss: 0.1369
Epoch 4/10
794/794 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.9860 - loss: 0.0655 - val_accuracy: 0.9697 - val_loss: 0.1330
Epoch 5/10
794/794 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.9868 - loss: 0.0480 - val_accuracy: 0.9692 - val_loss: 0.1555
Epoch 6/10
794/794 ━━━━━━━━━━━━━━━━━━━━ 41s 29ms/step - accuracy: 0.9921 - loss: 0.0346 - val_accuracy: 0.9636 - val_loss: 0.2036
Epoch 7/10
794/794 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.9946 - loss: 0.0220 - val_accuracy: 0.9819 - val_loss: 0.0886
Epoch 8/10
794/794 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.9928 - loss: 0.0268 - 

In [19]:
loss, acc = model.evaluate(X_dev, y_dev)
print("DEV accuracy:", acc)


777/777 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9629 - loss: 0.3615
DEV accuracy: 0.9634519219398499


In [21]:
model.save("model/deepfake_cnn.h5")
